Compute model evidence, P(D|M), for all models developped in this project.

FEV1, 2-day FEV1 FEF, long model

In [1]:
import concurrent.futures
from itertools import repeat

import numpy as np
import pandas as pd

import data.breathe_data as bd
import data.helpers as dh
import inf_cutset_conditioning.cutset_cond_algs_learn_ar_change as cca_ar_change
import inf_cutset_conditioning.cutset_cond_algs_learn_ar_change_noo2sat as cca_ar_change_noo2sat
import model_validation.model_evidence as me

In [2]:
df = bd.load_meas_from_excel("BR_O2_FEV1_FEF2575_conservative_smoothing_with_idx")

INFO:root:* Checking for same day measurements *


In [3]:
# P(D|M) on the strict 30-day long model (excluding ID below 30 days)
# Without FEF25-75, log_p_S_given_D = -49.08053225
# With FEV1 and FEF25-75, log_p_S_given_D = -148.65322444

In [3]:
# Reduce dataset to the last 30-day sequences

ndays = 20
df20 = pd.DataFrame(columns=df.columns)
for id in df.ID.unique():
    df_pre, start_idx, end_idx = dh.find_longest_conseq_sequence(
        df[df.ID == id], n_missing_days_allowed=1
    )

    dftmp = df_pre.tail(ndays).reset_index()

    if len(dftmp) < ndays:
        # print(f"Skipping ID {id}, n entries < {ndays} days")
        continue

    df20 = pd.concat([df20, dftmp])

/var/folders/zq/v2r6yn111s3gpdf8lzf72xvw0000gn/T/ipykernel_39139/546156911.py:16: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df20 = pd.concat([df20, dftmp])


In [4]:
df20.ID.nunique()

83

# Longitudinal model

In [ ]:
if __name__ == "__main__":
    with concurrent.futures.ProcessPoolExecutor() as executor:
        results = list(
            executor.map(me.process_id_long_model, df20.ID.unique()[0:1], repeat(df20))
        )
print(results)

101 - Time for 20 entries: 5.61 s
[-93.56807729406881]


# 2 day FEV1, FEF25-75 model

In [5]:
df_rmax_rows = (
    df.sort_values(by=["ecFEV1", "ecFEF2575", "O2 Saturation"], ascending=False)
    .groupby("ID")
    .agg(lambda df: df.head(1))
    .reset_index()
)
df_rmax_rows = df_rmax_rows[df_rmax_rows.ID.isin(df20.ID.unique())]

In [6]:
def process_data_no_interconnections(df):
    """
    Running this for one day with FEV1 and FEF gives same result than 
    the process_data_AR_through_time function
    """
    ar_prior = "breathe (2 days model, ecFEV1 addmultnoise, ecFEF25-75)"
    n_missing_days_allowed = 1
    ecfev1_noise_model_suffix = "_std_add_mult_ecfev1"
    fef2575_cpt_suffix = "_ecfev1_2_days_model_add_mult_noise"

    log_p_S_given_D, res_dict = (
        cca_ar_change_noo2sat.run_long_noise_model_no_ar_interconnections(
            df,
            ar_prior=ar_prior,
            ecfev1_noise_model_suffix=ecfev1_noise_model_suffix,
            fef2575_cpt_suffix=fef2575_cpt_suffix,
            debug=False,
        )
    )
    return log_p_S_given_D


def process_id_2day_fev1_fef_model(id, df20, df_rmax_rows):
    """
    To get results for the two days model, I run the process_id_longitudinal_data n times,
    each time adding the rmax FEV1 and rmax FEF25-75 as a second day.
    """
    df_for_ID = df20[df20.ID == id]
    df_rmax_row = df_rmax_rows[df_rmax_rows.ID == id]

    log_p_S_given_D = []
    for i, _ in df_for_ID.iterrows():
        dftmp = df_for_ID.iloc[i : i + 1]
        dftmp = pd.concat([df_rmax_row, dftmp]).reset_index()
        log_p_S_given_Di = process_data_no_interconnections(dftmp)
        log_p_S_given_D.append(log_p_S_given_Di)
    return np.sum(log_p_S_given_D)


process_id_2day_fev1_fef_model("101", df20, df_rmax_rows)

101 - Time for 2 entries: 0.98 s
101 - Time for 2 entries: 1.03 s
101 - Time for 2 entries: 1.08 s
101 - Time for 2 entries: 1.10 s
101 - Time for 2 entries: 1.02 s
101 - Time for 2 entries: 1.22 s
101 - Time for 2 entries: 0.91 s
101 - Time for 2 entries: 0.98 s
101 - Time for 2 entries: 0.92 s
101 - Time for 2 entries: 0.92 s
101 - Time for 2 entries: 1.07 s
101 - Time for 2 entries: 0.97 s
101 - Time for 2 entries: 0.92 s
101 - Time for 2 entries: 0.98 s
101 - Time for 2 entries: 0.98 s
101 - Time for 2 entries: 1.02 s
101 - Time for 2 entries: 0.92 s
101 - Time for 2 entries: 0.92 s
101 - Time for 2 entries: 0.92 s
101 - Time for 2 entries: 0.98 s


-138.37374487152357

In [ ]:
# with rmax FEV1 -276.9759478315757

# FEV1, FEF25-75 model

In [13]:
if __name__ == "__main__":
    with concurrent.futures.ProcessPoolExecutor() as executor:
        results = list(
            executor.map(
                me.process_id_fev1_fef_model, df20.ID.unique()[0:1], repeat(df20)
            )
        )

101 - Time for 1 entries: 0.49 s
101 - Time for 1 entries: 0.52 s
101 - Time for 1 entries: 0.55 s
101 - Time for 1 entries: 0.61 s
101 - Time for 1 entries: 0.57 s
101 - Time for 1 entries: 0.61 s
101 - Time for 1 entries: 0.49 s
101 - Time for 1 entries: 0.47 s
101 - Time for 1 entries: 0.47 s
101 - Time for 1 entries: 0.47 s
101 - Time for 1 entries: 0.47 s
101 - Time for 1 entries: 0.47 s
101 - Time for 1 entries: 0.47 s
101 - Time for 1 entries: 0.53 s
101 - Time for 1 entries: 0.50 s
101 - Time for 1 entries: 0.50 s
101 - Time for 1 entries: 0.45 s
101 - Time for 1 entries: 0.45 s
101 - Time for 1 entries: 0.55 s
101 - Time for 1 entries: 0.47 s


In [14]:
np.sum(results)

-138.91887114776512